# Comparing lag-based gradient boosting

> **Historical v1 analysis.** The modelling target in this workflow ends in September 2025; newer supplemental observations remain separate here. For the current 519-game forecast, see [Notebook 20](20_fixed_cohort_2027_forecast.ipynb). [Notebook 17](17_fixed_sources_integration.ipynb) begins the separate v2 workflow.

This step adds one ML candidate to the same historical forecasting task used for the baselines and smoothing models. Each forecast is issued once at its cutoff, with predicted activity fed into later steps.

## 1. Compare the six implementations

The table reports mean absolute error for leads 16–27. Lower is better within a category; absolute errors differ with category size.

In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
scores = pd.read_csv(ROOT / "reports/model_comparison/scores.csv")
long_lead = scores.loc[scores.window.eq("Leads 16 to 27")]
comparison = long_lead.pivot(index="category", columns="model", values="mae").round(2)
comparison


model,Damped seasonal,Damped trend,Gradient boosting,Last value,SES,Seasonal naive
category,,,,,,
Action,378208.72,280759.17,341764.52,259766.23,259766.23,332754.30
Indie,59388.53,72435.17,54346.87,66931.97,66931.97,101775.52
Massively Multiplayer,84749.09,116113.82,66380.81,64311.70,64311.70,75472.38
RPG,66660.63,84386.62,58620.57,57019.74,57019.74,83441.83
Strategy,64773.91,71705.70,39031.61,32157.78,34535.67,45987.08


## Interpretation

Gradient boosting improves Indie MAE to 54,346.87, approximately 18.8% below last value and 8.5% below damped seasonal smoothing. Last value remains ahead for the other four categories. This supports keeping simple forecasts in contention while examining when each model succeeds.

## 2. Inspect variation across cutoffs

The aggregate Indie improvement should be read beside the individual origins. These windows overlap, so the rows are not independent experiments.

In [2]:
fold_scores = pd.read_csv(ROOT / "reports/model_comparison/fold_scores.csv")
indie_folds = fold_scores.loc[fold_scores.category.eq("Indie")].pivot(index="origin", columns="model", values="mae").round(2)
indie_folds


model,Damped seasonal,Damped trend,Gradient boosting,Last value,SES,Seasonal naive
origin,,,,,,
2020-06-01,60559.20,73589.78,63895.06,65673.27,65673.27,204655.90
2021-06-01,36608.58,52092.03,52218.69,95312.47,95312.47,85210.27
2022-06-01,94286.51,98368.05,56386.36,42817.90,42817.90,73922.06
2023-06-01,46099.82,65690.82,44887.38,63924.23,63924.23,43313.85


## Model comparison findings

This stage compared all planned model families: last value, seasonal naive, exponential smoothing and gradient boosting. Selection was left open pending fold-stability and evaluation-scope review. The v1 target series ends in September 2025.

Recorded evidence: [combined model scores](../reports/model_comparison/scores.csv).